In [3]:
import pandas as pd
import sqlite3
from pathlib import Path

In [15]:
tale = """
The Three Cows
England
There was a farmer, and he had three cows, fine fat beauties they were. One was called Facey, the other Diamond, and the third Beauty. One morning he went into his cowshed, and there he found Facey so thin that the wind would have blown her away. Her skin hung loose about her, all her flesh was gone, and she stared out of her great eyes as though she'd seen a ghost; and what was more, the fireplace in the kitchen was one great pile of wood-ash. Well, he was bothered with it; he could not see how all this had come about.
Next morning his wife went out to the shed, and see! Diamond was for all the world as wisht a looking creature as Facey -- nothing but a bag of bones, all the flesh gone, and half a rick of wood was gone, too; but the fireplace was piled up three feet high with white wood ashes. The farmer determined to watch the third night; so he hid in a closet which opened out of the parlor, and he left the door just ajar, that he might see what passed.

Tick, tick went the clock, and the farmer was nearly tired of waiting; he had to bite his little finger to keep himself awake, when suddenly the door of his house flew open, and in rushed maybe a thousand pixies, laughing and dancing and dragging at Beauty's halter till they had brought the cow into the middle of the room. The farmer really thought he should have died with fright, and so perhaps he would, had not curiosity kept him alive.

Tick, tick went the clock, but he did not hear it now. He was too intent staring at the pixies and his last beautiful cow. He saw them throw her down, fall on her, and kill her; then with their knives they ripped her open, and flayed her as clean as a whistle. Then out ran some of the little people and brought in firewood and made a roaring blaze on the hearth, and there they cooked the flesh of the cow. They baked and they boiled, they stewed and they fried.

"Take care," cried one, who seemed to be the king. "Let no bone be broken."

Well, when they had all eaten, and had devoured every scrap of beef on the cow, they began playing games with the bones, tossing them one to another. One little leg bone fell close to the closet door, and the farmer was so afraid lest the pixies should come there and find him in their search for the bone, that he put out his hand and drew it in to him.

Then he saw the king stand on the table and say, "Gather the bones!"

Round and round flew the imps, picking up the bones.

"Arrange them," said the king; and they placed them all in their proper positions in the hide of the cow.

Then they folded the skin over them, and the king struck the heap of bone and skin with his rod. Whisht! Up sprang the cow and lowed dismally. It was alive again; but alas! as the pixies dragged it back to its stall, it halted in the off forefoot, for a bone was missing.


The cock crew,
Away they flew.
And the farmer crept trembling to bed.

"""

In [24]:
# Definição dos Sete Enredos Básicos de Christopher Booker
SEVEN_BASIC_PLOTS = {
    "overcoming_the_monster": {
        "name": "Overcoming the Monster",
        "description": "O herói enfrenta e derrota uma força ameaçadora (monstro, vilão, etc.)"
    },
    "rags_to_riches": {
        "name": "Rags to Riches",
        "description": "O protagonista começa em circunstâncias humildes e alcança sucesso/riqueza"
    },
    "the_quest": {
        "name": "The Quest",
        "description": "O herói parte em uma jornada para alcançar um objetivo importante"
    },
    "voyage_and_return": {
        "name": "Voyage and Return",
        "description": "O protagonista viaja para um lugar estranho e retorna transformado"
    },
    "comedy": {
        "name": "Comedy",
        "description": "Confusão e mal-entendidos são resolvidos, levando a um final feliz"
    },
    "tragedy": {
        "name": "Tragedy",
        "description": "O protagonista tem uma falha fatal que leva à sua queda"
    },
    "rebirth": {
        "name": "Rebirth",
        "description": "O protagonista é transformado ou redimido após eventos negativos"
    }
}

In [22]:
from pydantic import BaseModel, Field
from enum import Enum
from langchain_openai import ChatOpenAI

# Crie um Enum com as chaves do dicionário
class PlotType(str, Enum):
    overcoming_the_monster = "overcoming_the_monster"
    rags_to_riches = "rags_to_riches"
    the_quest = "the_quest"
    voyage_and_return = "voyage_and_return"
    comedy = "comedy"
    tragedy = "tragedy"
    rebirth = "rebirth"

# Defina o schema de classificação
class PlotClassification(BaseModel):
    plot_type: PlotType = Field(description="O tipo de enredo identificado")
    reasoning: str = Field(description="Explicação do motivo da classificação")
    confidence: float = Field(description="Confiança da classificação (0-1)")

# Configure o modelo
llm = ChatOpenAI(model="gpt-4o")
classificador = llm.with_structured_output(PlotClassification)

# Use o classificador com as descrições do dicionário no prompt
prompt = f"""Classifique o enredo da história nas seguintes categorias:
{chr(10).join(f"- {k}: {v['description']}" for k, v in SEVEN_BASIC_PLOTS.items())}

História: {{texto}}"""

resultado = classificador.invoke(prompt.format(texto=tale))
print(resultado.plot_type)  # PlotType.overcoming_the_monster

PlotType.voyage_and_return


In [23]:
resultado

PlotClassification(plot_type=<PlotType.voyage_and_return: 'voyage_and_return'>, reasoning="While the initial setup includes a mysterious depletion of the farm animals, the core of the story revolves around the farmer's experience with a mystical realm represented by pixies. He embarks on this 'voyage' from the safety and normalcy of his life into a surreal, terrifying, and transformative encounter within his own home. The 'journey' is internal and situational—a temporary and surreal immersion into the world of the pixies that changes his perspective on the reality he understood. Ultimately, he returns 'home'—to the life he knew prior, albeit with new knowledge and experience regarding the pixies and their capability to bring creatures back to life, albeit flawed. The transformative aspect is the key identifier of the 'voyage and return' plot.", confidence=0.75)

In [ ]:

from pydantic import BaseModel, Field
from enum import Enum
from typing import List, Optional
from langchain_openai import ChatOpenAI

# Definição dos 7 personagens de Propp
PROPP_CHARACTERS = {
    "villain": {
        "name": "Villain (Antagonista)",
        "description": "O vilão que luta contra o herói, causa dano ou prejudica alguém"
    },
    "donor": {
        "name": "Donor (Doador)",
        "description": "Prepara o herói ou lhe dá algum objeto mágico/ajuda"
    },
    "helper": {
        "name": "Helper (Auxiliar)",
        "description": "Ajuda o herói em sua busca, geralmente após receber o objeto mágico"
    },
    "princess": {
        "name": "Princess (Princesa) and her Father",
        "description": "O objeto da busca do herói, frequentemente oferecida como recompensa; seu pai frequentemente propõe tarefas difíceis"
    },
    "dispatcher": {
        "name": "Dispatcher (Mandante)",
        "description": "Envia o herói em sua missão através de uma ordem ou pedido"
    },
    "hero": {
        "name": "Hero (Herói)",
        "description": "O protagonista que reage ao doador, parte em busca e se casa com a princesa"
    },
    "false_hero": {
        "name": "False Hero (Falso Herói)",
        "description": "Toma crédito pelas ações do herói ou tenta casar com a princesa"
    }
}

# Enums para Propp
class ProppCharacterType(str, Enum):
    villain = "villain"
    donor = "donor"
    helper = "helper"
    princess = "princess"
    dispatcher = "dispatcher"
    hero = "hero"
    false_hero = "false_hero"

# Schema para identificação de personagem individual
class ProppCharacter(BaseModel):
    character_name: str = Field(description="Nome ou descrição do personagem na história")
    character_type: ProppCharacterType = Field(description="Tipo de personagem segundo Propp")
    reasoning: str = Field(description="Justificativa para a classificação deste personagem")
    key_actions: List[str] = Field(description="Ações principais que justificam a classificação")

# Schema para classificação completa de Propp
class ProppAnalysis(BaseModel):
    characters: List[ProppCharacter] = Field(description="Lista de personagens identificados")
    narrative_structure: str = Field(description="Descrição da estrutura narrativa segundo Propp")
    confidence: float = Field(description="Confiança geral da análise (0-1)")

# Schema combinado: Booker + Propp
class CompleteStoryAnalysis(BaseModel):
    # Análise de Booker
    booker_plot: PlotType = Field(description="Tipo de enredo segundo Booker")
    booker_reasoning: str = Field(description="Justificativa da classificação de Booker")
    booker_confidence: float = Field(description="Confiança da classificação de Booker (0-1)")
    
    # Análise de Propp
    propp_characters: List[ProppCharacter] = Field(description="Personagens identificados segundo Propp")
    propp_narrative: str = Field(description="Estrutura narrativa segundo Propp")
    propp_confidence: float = Field(description="Confiança da análise de Propp (0-1)")
    
    # Análise integrada
    integrated_analysis: str = Field(description="Como as análises de Booker e Propp se complementam nesta história")

# Configure o modelo
llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# Classificador apenas de Propp
propp_classifier = llm.with_structured_output(ProppAnalysis)

# Classificador completo (Booker + Propp)
complete_classifier = llm.with_structured_output(CompleteStoryAnalysis)

# Prompt para análise de Propp
propp_prompt = f"""Analise os personagens da história segundo a morfologia de Vladimir Propp.

Personagens de Propp:
{chr(10).join(f"- {k}: {v['description']}" for k, v in PROPP_CHARACTERS.items())}

Para cada personagem identificado na história, determine:
1. Qual tipo de personagem de Propp ele representa
2. Quais ações específicas justificam essa classificação
3. Como ele contribui para a estrutura narrativa

História: {{texto}}

Importante: 
- Um personagem pode desempenhar múltiplos papéis
- Nem todos os 7 tipos precisam estar presentes
- Foque nas funções narrativas, não apenas em características superficiais
"""

# Prompt para análise completa
complete_prompt = f"""Realize uma análise literária completa da história, combinando:

1. ANÁLISE DE BOOKER - Classifique segundo os Sete Enredos Básicos:
{chr(10).join(f"- {k}: {v['description']}" for k, v in SEVEN_BASIC_PLOTS.items())}

2. ANÁLISE DE PROPP - Identifique os personagens segundo a morfologia:
{chr(10).join(f"- {k}: {v['description']}" for k, v in PROPP_CHARACTERS.items())}

3. ANÁLISE INTEGRADA - Explique como as duas perspectivas se complementam para entender a história.

História: {{texto}}

Seja preciso e detalhado em suas análises.
"""

# Função para análise apenas de Propp
def analyze_propp(text: str) -> ProppAnalysis:
    """Analisa personagens segundo Propp"""
    result = propp_classifier.invoke(propp_prompt.format(texto=text))
    return result

# Função para análise completa
def analyze_complete(text: str) -> CompleteStoryAnalysis:
    """Análise completa: Booker + Propp"""
    result = complete_classifier.invoke(complete_prompt.format(texto=text))
    return result

# Função para exibir análise de Propp
def display_propp_analysis(analysis: ProppAnalysis):
    """Exibe análise de Propp de forma formatada"""
    print("=" * 80)
    print("ANÁLISE DE PERSONAGENS SEGUNDO VLADIMIR PROPP")
    print("=" * 80)
    
    print(f"\n📊 Confiança Geral: {analysis.confidence:.2%}")
    print(f"\n📖 Estrutura Narrativa:\n{analysis.narrative_structure}")
    
    print(f"\n👥 Personagens Identificados ({len(analysis.characters)}):")
    print("-" * 80)
    
    for i, char in enumerate(analysis.characters, 1):
        print(f"\n{i}. {char.character_name}")
        print(f"   Tipo: {char.character_type.value.replace('_', ' ').title()}")
        print(f"   Justificativa: {char.reasoning}")
        print(f"   Ações-chave:")
        for action in char.key_actions:
            print(f"      • {action}")
    
    print("=" * 80)

# Função para exibir análise completa
def display_complete_analysis(analysis: CompleteStoryAnalysis):
    """Exibe análise completa de forma formatada"""
    print("=" * 80)
    print("ANÁLISE LITERÁRIA COMPLETA: BOOKER + PROPP")
    print("=" * 80)
    
    # Seção Booker
    print("\n" + "="*80)
    print("📚 ANÁLISE DE BOOKER - OS SETE ENREDOS BÁSICOS")
    print("="*80)
    print(f"\nEnredo: {analysis.booker_plot.value.replace('_', ' ').title()}")
    print(f"Confiança: {analysis.booker_confidence:.2%}")
    print(f"\nJustificativa:\n{analysis.booker_reasoning}")
    
    # Seção Propp
    print("\n" + "="*80)
    print("👥 ANÁLISE DE PROPP - MORFOLOGIA DOS PERSONAGENS")
    print("="*80)
    print(f"\nConfiança: {analysis.propp_confidence:.2%}")
    print(f"\nEstrutura Narrativa:\n{analysis.propp_narrative}")
    
    print(f"\nPersonagens Identificados ({len(analysis.propp_characters)}):")
    print("-" * 80)
    
    for i, char in enumerate(analysis.propp_characters, 1):
        print(f"\n{i}. {char.character_name}")
        print(f"   Tipo: {char.character_type.value.replace('_', ' ').title()}")
        print(f"   Justificativa: {char.reasoning}")
        print(f"   Ações-chave:")
        for action in char.key_actions:
            print(f"      • {action}")
    
    # Análise Integrada
    print("\n" + "="*80)
    print("🔗 ANÁLISE INTEGRADA")
    print("="*80)
    print(f"\n{analysis.integrated_analysis}")
    print("\n" + "="*80)

# Exemplo de uso - Análise apenas de Propp
print("ANÁLISE APENAS DE PROPP:")
print("-" * 80)
propp_result = analyze_propp(tale)
display_propp_analysis(propp_result)

print("\n\n")

# Exemplo de uso - Análise completa
print("ANÁLISE COMPLETA (BOOKER + PROPP):")
print("-" * 80)
complete_result = analyze_complete(tale)
display_complete_analysis(complete_result)

In [ ]:
### Question:
como posso usar uma LLM para classificar um texto segundo Os Sete Enredos Básicos (Christopher Booker)?

### Answer:
Para classificar um texto segundo os Sete Enredos Básicos de Christopher Booker usando uma LLM, você pode criar um sistema que analisa o texto e identifica qual dos sete arquétipos narrativos ele representa. Aqui está uma solução completa:

```python
# File: /Users/gustavosarti/Work/code/xer_persona/xer_persona/notebooks/classificar_propp.ipynb

from openai import OpenAI
from anthropic import Anthropic
import os
from typing import Literal

# Definição dos Sete Enredos Básicos de Christopher Booker
SEVEN_BASIC_PLOTS = {
    "overcoming_the_monster": {
        "name": "Overcoming the Monster",
        "description": "O herói enfrenta e derrota uma força ameaçadora (monstro, vilão, etc.)"
    },
    "rags_to_riches": {
        "name": "Rags to Riches",
        "description": "O protagonista começa em circunstâncias humildes e alcança sucesso/riqueza"
    },
    "the_quest": {
        "name": "The Quest",
        "description": "O herói parte em uma jornada para alcançar um objetivo importante"
    },
    "voyage_and_return": {
        "name": "Voyage and Return",
        "description": "O protagonista viaja para um lugar estranho e retorna transformado"
    },
    "comedy": {
        "name": "Comedy",
        "description": "Confusão e mal-entendidos são resolvidos, levando a um final feliz"
    },
    "tragedy": {
        "name": "Tragedy",
        "description": "O protagonista tem uma falha fatal que leva à sua queda"
    },
    "rebirth": {
        "name": "Rebirth",
        "description": "O protagonista é transformado ou redimido após eventos negativos"
    }
}

def classify_story_openai(text: str, model: str = "gpt-4") -> dict:
    """
    Classifica uma história usando OpenAI GPT
    """
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    
    # Criar descrição dos enredos para o prompt
    plots_description = "\n".join([
        f"{i+1}. {plot['name']}: {plot['description']}"
        for i, plot in enumerate(SEVEN_BASIC_PLOTS.values())
    ])
    
    prompt = f"""Analise a seguinte história e classifique-a segundo os Sete Enredos Básicos de Christopher Booker:

{plots_description}

História:
{text}

Por favor, identifique:
1. Qual dos sete enredos básicos melhor descreve esta história
2. Forneça uma justificativa detalhada para sua escolha
3. Indique elementos específicos da história que suportam sua classificação
4. Se houver elementos de múltiplos enredos, mencione-os

Responda em formato JSON com as seguintes chaves:
- primary_plot: o enredo principal (use o nome em inglês)
- confidence: nível de confiança (0-100)
- justification: justificativa detalhada
- key_elements: lista de elementos-chave que suportam a classificação
- secondary_plots: lista de enredos secundários, se houver
"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Você é um especialista em análise literária e narratologia, com profundo conhecimento dos Sete Enredos Básicos de Christopher Booker."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.3
    )
    
    import json
    result = json.loads(response.choices[0].message.content)
    return result

def classify_story_anthropic(text: str, model: str = "claude-3-5-sonnet-20241022") -> dict:
    """
    Classifica uma história usando Anthropic Claude
    """
    client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    
    plots_description = "\n".join([
        f"{i+1}. {plot['name']}: {plot['description']}"
        for i, plot in enumerate(SEVEN_BASIC_PLOTS.values())
    ])
    
    prompt = f"""Analise a seguinte história e classifique-a segundo os Sete Enredos Básicos de Christopher Booker:

{plots_description}

História:
{text}

Por favor, identifique:
1. Qual dos sete enredos básicos melhor descreve esta história
2. Forneça uma justificativa detalhada para sua escolha
3. Indique elementos específicos da história que suportam sua classificação
4. Se houver elementos de múltiplos enredos, mencione-os

Responda em formato JSON com as seguintes chaves:
- primary_plot: o enredo principal (use o nome em inglês)
- confidence: nível de confiança (0-100)
- justification: justificativa detalhada
- key_elements: lista de elementos-chave que suportam a classificação
- secondary_plots: lista de enredos secundários, se houver
"""
    
    response = client.messages.create(
        model=model,
        max_tokens=2000,
        temperature=0.3,
        system="Você é um especialista em análise literária e narratologia, com profundo conhecimento dos Sete Enredos Básicos de Christopher Booker.",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    import json
    # Extrair JSON da resposta
    content = response.content[0].text
    # Tentar encontrar JSON no texto
    start = content.find('{')
    end = content.rfind('}') + 1
    json_str = content[start:end]
    result = json.loads(json_str)
    return result

def display_classification(result: dict):
    """
    Exibe os resultados da classificação de forma formatada
    """
    print("=" * 80)
    print("CLASSIFICAÇÃO SEGUNDO OS SETE ENREDOS BÁSICOS")
    print("=" * 80)
    print(f"\n📚 Enredo Principal: {result['primary_plot']}")
    print(f"📊 Confiança: {result['confidence']}%")
    print(f"\n💡 Justificativa:\n{result['justification']}")
    print(f"\n🔑 Elementos-chave:")
    for element in result['key_elements']:
        print(f"  • {element}")
    
    if result.get('secondary_plots'):
        print(f"\n📖 Enredos Secundários:")
        for plot in result['secondary_plots']:
            print(f"  • {plot}")
    print("=" * 80)

# Exemplo de uso com a história fornecida
if __name__ == "__main__":
    # Escolha o provedor: 'openai' ou 'anthropic'
    provider = 'anthropic'  # ou 'openai'
    
    if provider == 'openai':
        result = classify_story_openai(tale)
    else:
        result = classify_story_anthropic(tale)
    
    display_classification(result)
```

Para usar com um banco de dados SQLite e classificar múltiplas histórias:

```python
# File: /Users/gustavosarti/Work/code/xer_persona/xer_persona/notebooks/classificar_propp.ipynb

def classify_stories_from_db(db_path: str, provider: Literal['openai', 'anthropic'] = 'anthropic'):
    """
    Classifica histórias de um banco de dados SQLite
    """
    conn = sqlite3.connect(db_path)
    
    # Ler histórias do banco
    df = pd.read_sql_query("SELECT id, title, text FROM stories", conn)
    
    results = []
    
    for idx, row in df.iterrows():
        print(f"\nProcessando: {row['title']} ({idx+1}/{len(df)})")
        
        try:
            if provider == 'openai':
                classification = classify_story_openai(row['text'])
            else:
                classification = classify_story_anthropic(row['text'])
            
            results.append({
                'story_id': row['id'],
                'title': row['title'],
                'primary_plot': classification['primary_plot'],
                'confidence': classification['confidence'],
                'justification': classification['justification'],
                'key_elements': ', '.join(classification['key_elements']),
                'secondary_plots': ', '.join(classification.get('secondary_plots', []))
            })
            
        except Exception as e:
            print(f"Erro ao processar {row['title']}: {e}")
            continue
    
    # Criar DataFrame com resultados
    results_df = pd.DataFrame(results)
    
    # Salvar no banco de dados
    results_df.to_sql('story_classifications', conn, if_exists='replace', index=False)
    
    conn.close()
    
    return results_df

# Exemplo de uso
db_path = Path("../../data/contos.sqlite")
if db_path.exists():
    results_df = classify_stories_from_db(str(db_path), provider='anthropic')
    print("\n📊 Resumo das Classificações:")
    print(results_df['primary_plot'].value_counts())
```

Para análise mais detalhada com visualizações:

```python
# File: /Users/gustavosarti/Work/code/xer_persona/xer_persona/notebooks/classificar_propp.ipynb

import matplotlib.pyplot as plt
import seaborn as sns

def analyze_classifications(results_df: pd.DataFrame):
    """
    Analisa e visualiza os resultados das classificações
    """
    # Configurar estilo
    sns.set_style("whitegrid")
    plt.figure(figsize=(12, 6))
    
    # Gráfico de distribuição dos enredos
    plt.subplot(1, 2, 1)
    plot_counts = results_df['primary_plot'].value_counts()
    plt.bar(range(len(plot_counts)), plot_counts.values)
    plt.xticks(range(len(plot_counts)), plot_counts.index, rotation=45, ha='right')
    plt.title('Distribuição dos Enredos Básicos')
    plt.ylabel('Número de Histórias')
    
    # Gráfico de confiança média por enredo
    plt.subplot(1, 2, 2)
    confidence_by_plot = results_df.groupby('primary_plot')['confidence'].mean()
    plt.bar(range(len(confidence_by_plot)), confidence_by_plot.values)
    plt.xticks(range(len(confidence_by_plot)), confidence_by_plot.index, rotation=45, ha='right')
    plt.title('Confiança Média por Enredo')
    plt.ylabel('Confiança (%)')
    
    plt.tight_layout()
    plt.show()
    
    # Estatísticas descritivas
    print("\n📈 Estatísticas:")
    print(f"Total de histórias classificadas: {len(results_df)}")
    print(f"Confiança média: {results_df['confidence'].mean():.2f}%")
    print(f"Confiança mínima: {results_df['confidence'].min():.2f}%")
    print(f"Confiança máxima: {results_df['confidence'].max():.2f}%")

# Uso
if 'results_df' in locals():
    analyze_classifications(results_df)
```

**Principais características desta solução:**

1. **Suporte para múltiplos provedores**: OpenAI e Anthropic
2. **Classificação estruturada**: Retorna JSON com enredo principal, confiança, justificativa e elementos-chave
3. **Integração com banco de dados**: Processa múltiplas histórias de um SQLite
4. **Visualizações**: Gráficos para análise dos resultados
5. **Tratamento de erros**: Continua processamento mesmo se uma história falhar

Para usar, você precisa:
1. Instalar as dependências: `pip install openai anthropic pandas matplotlib seaborn`
2. Configurar as variáveis de ambiente com suas API keys
3. Executar a classificação escolhendo o provedor

### Question:
Além de Booker, gostaria de fazer uma classificação considerando os personagens descritos por Propp

### Answer:
Excelente! Vou adicionar a classificação de personagens segundo Vladimir Propp. Propp identificou 7 tipos de personagens (dramatis personae) em contos folclóricos. Aqui está a solução completa:

```python
# File: /Users/gustavosarti/Work/code/xer_persona/xer_persona/notebooks/classificar_contos.ipynb

from pydantic import BaseModel, Field
from enum import Enum
from typing import List, Optional
from langchain_openai import ChatOpenAI

# Definição dos 7 personagens de Propp
PROPP_CHARACTERS = {
    "villain": {
        "name": "Villain (Antagonista)",
        "description": "O vilão que luta contra o herói, causa dano ou prejudica alguém"
    },
    "donor": {
        "name": "Donor (Doador)",
        "description": "Prepara o herói ou lhe dá algum objeto mágico/ajuda"
    },
    "helper": {
        "name": "Helper (Auxiliar)",
        "description": "Ajuda o herói em sua busca, geralmente após receber o objeto mágico"
    },
    "princess": {
        "name": "Princess (Princesa) and her Father",
        "description": "O objeto da busca do herói, frequentemente oferecida como recompensa; seu pai frequentemente propõe tarefas difíceis"
    },
    "dispatcher": {
        "name": "Dispatcher (Mandante)",
        "description": "Envia o herói em sua missão através de uma ordem ou pedido"
    },
    "hero": {
        "name": "Hero (Herói)",
        "description": "O protagonista que reage ao doador, parte em busca e se casa com a princesa"
    },
    "false_hero": {
        "name": "False Hero (Falso Herói)",
        "description": "Toma crédito pelas ações do herói ou tenta casar com a princesa"
    }
}

# Enums para Propp
class ProppCharacterType(str, Enum):
    villain = "villain"
    donor = "donor"
    helper = "helper"
    princess = "princess"
    dispatcher = "dispatcher"
    hero = "hero"
    false_hero = "false_hero"

# Schema para identificação de personagem individual
class ProppCharacter(BaseModel):
    character_name: str = Field(description="Nome ou descrição do personagem na história")
    character_type: ProppCharacterType = Field(description="Tipo de personagem segundo Propp")
    reasoning: str = Field(description="Justificativa para a classificação deste personagem")
    key_actions: List[str] = Field(description="Ações principais que justificam a classificação")

# Schema para classificação completa de Propp
class ProppAnalysis(BaseModel):
    characters: List[ProppCharacter] = Field(description="Lista de personagens identificados")
    narrative_structure: str = Field(description="Descrição da estrutura narrativa segundo Propp")
    confidence: float = Field(description="Confiança geral da análise (0-1)")

# Schema combinado: Booker + Propp
class CompleteStoryAnalysis(BaseModel):
    # Análise de Booker
    booker_plot: PlotType = Field(description="Tipo de enredo segundo Booker")
    booker_reasoning: str = Field(description="Justificativa da classificação de Booker")
    booker_confidence: float = Field(description="Confiança da classificação de Booker (0-1)")
    
    # Análise de Propp
    propp_characters: List[ProppCharacter] = Field(description="Personagens identificados segundo Propp")
    propp_narrative: str = Field(description="Estrutura narrativa segundo Propp")
    propp_confidence: float = Field(description="Confiança da análise de Propp (0-1)")
    
    # Análise integrada
    integrated_analysis: str = Field(description="Como as análises de Booker e Propp se complementam nesta história")

# Configure o modelo
llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# Classificador apenas de Propp
propp_classifier = llm.with_structured_output(ProppAnalysis)

# Classificador completo (Booker + Propp)
complete_classifier = llm.with_structured_output(CompleteStoryAnalysis)

# Prompt para análise de Propp
propp_prompt = f"""Analise os personagens da história segundo a morfologia de Vladimir Propp.

Personagens de Propp:
{chr(10).join(f"- {k}: {v['description']}" for k, v in PROPP_CHARACTERS.items())}

Para cada personagem identificado na história, determine:
1. Qual tipo de personagem de Propp ele representa
2. Quais ações específicas justificam essa classificação
3. Como ele contribui para a estrutura narrativa

História: {{texto}}

Importante: 
- Um personagem pode desempenhar múltiplos papéis
- Nem todos os 7 tipos precisam estar presentes
- Foque nas funções narrativas, não apenas em características superficiais
"""

# Prompt para análise completa
complete_prompt = f"""Realize uma análise literária completa da história, combinando:

1. ANÁLISE DE BOOKER - Classifique segundo os Sete Enredos Básicos:
{chr(10).join(f"- {k}: {v['description']}" for k, v in SEVEN_BASIC_PLOTS.items())}

2. ANÁLISE DE PROPP - Identifique os personagens segundo a morfologia:
{chr(10).join(f"- {k}: {v['description']}" for k, v in PROPP_CHARACTERS.items())}

3. ANÁLISE INTEGRADA - Explique como as duas perspectivas se complementam para entender a história.

História: {{texto}}

Seja preciso e detalhado em suas análises.
"""

# Função para análise apenas de Propp
def analyze_propp(text: str) -> ProppAnalysis:
    """Analisa personagens segundo Propp"""
    result = propp_classifier.invoke(propp_prompt.format(texto=text))
    return result

# Função para análise completa
def analyze_complete(text: str) -> CompleteStoryAnalysis:
    """Análise completa: Booker + Propp"""
    result = complete_classifier.invoke(complete_prompt.format(texto=text))
    return result

# Função para exibir análise de Propp
def display_propp_analysis(analysis: ProppAnalysis):
    """Exibe análise de Propp de forma formatada"""
    print("=" * 80)
    print("ANÁLISE DE PERSONAGENS SEGUNDO VLADIMIR PROPP")
    print("=" * 80)
    
    print(f"\n📊 Confiança Geral: {analysis.confidence:.2%}")
    print(f"\n📖 Estrutura Narrativa:\n{analysis.narrative_structure}")
    
    print(f"\n👥 Personagens Identificados ({len(analysis.characters)}):")
    print("-" * 80)
    
    for i, char in enumerate(analysis.characters, 1):
        print(f"\n{i}. {char.character_name}")
        print(f"   Tipo: {char.character_type.value.replace('_', ' ').title()}")
        print(f"   Justificativa: {char.reasoning}")
        print(f"   Ações-chave:")
        for action in char.key_actions:
            print(f"      • {action}")
    
    print("=" * 80)

# Função para exibir análise completa
def display_complete_analysis(analysis: CompleteStoryAnalysis):
    """Exibe análise completa de forma formatada"""
    print("=" * 80)
    print("ANÁLISE LITERÁRIA COMPLETA: BOOKER + PROPP")
    print("=" * 80)
    
    # Seção Booker
    print("\n" + "="*80)
    print("📚 ANÁLISE DE BOOKER - OS SETE ENREDOS BÁSICOS")
    print("="*80)
    print(f"\nEnredo: {analysis.booker_plot.value.replace('_', ' ').title()}")
    print(f"Confiança: {analysis.booker_confidence:.2%}")
    print(f"\nJustificativa:\n{analysis.booker_reasoning}")
    
    # Seção Propp
    print("\n" + "="*80)
    print("👥 ANÁLISE DE PROPP - MORFOLOGIA DOS PERSONAGENS")
    print("="*80)
    print(f"\nConfiança: {analysis.propp_confidence:.2%}")
    print(f"\nEstrutura Narrativa:\n{analysis.propp_narrative}")
    
    print(f"\nPersonagens Identificados ({len(analysis.propp_characters)}):")
    print("-" * 80)
    
    for i, char in enumerate(analysis.propp_characters, 1):
        print(f"\n{i}. {char.character_name}")
        print(f"   Tipo: {char.character_type.value.replace('_', ' ').title()}")
        print(f"   Justificativa: {char.reasoning}")
        print(f"   Ações-chave:")
        for action in char.key_actions:
            print(f"      • {action}")
    
    # Análise Integrada
    print("\n" + "="*80)
    print("🔗 ANÁLISE INTEGRADA")
    print("="*80)
    print(f"\n{analysis.integrated_analysis}")
    print("\n" + "="*80)

# Exemplo de uso - Análise apenas de Propp
print("ANÁLISE APENAS DE PROPP:")
print("-" * 80)
propp_result = analyze_propp(tale)
display_propp_analysis(propp_result)

print("\n\n")

# Exemplo de uso - Análise completa
print("ANÁLISE COMPLETA (BOOKER + PROPP):")
print("-" * 80)
complete_result = analyze_complete(tale)
display_complete_analysis(complete_result)
```

Agora vamos criar funções para processar múltiplas histórias do banco de dados:

```python
# File: /Users/gustavosarti/Work/code/xer_persona/xer_persona/notebooks/classificar_contos.ipynb

import json
from typing import Dict, Any

def process_stories_from_db(
    db_path: str,
    analysis_type: str = "complete",  # "propp", "booker", ou "complete"
    limit: Optional[int] = None
) -> pd.DataFrame:
    """
    Processa histórias do banco de dados com análise Propp, Booker ou completa
    
    Args:
        db_path: Caminho para o banco SQLite
        analysis_type: Tipo de análise ("propp", "booker", "complete")
        limit: Número máximo de histórias a processar (None = todas)
    """
    conn = sqlite3.connect(db_path)
    
    # Ler histórias
    query = "SELECT id, title, text FROM stories"
    if limit:
        query += f" LIMIT {limit}"
    
    df = pd.read_sql_query(query, conn)
    
    results = []
    
    for idx, row in df.iterrows():
        print(f"\n{'='*80}")
        print(f"Processando: {row['title']} ({idx+1}/{len(df)})")
        print(f"{'='*80}")
        
        try:
            if analysis_type == "propp":
                analysis = analyze_propp(row['text'])
                result = {
                    'story_id': row['id'],
                    'title': row['title'],
                    'propp_confidence': analysis.confidence,
                    'propp_narrative': analysis.narrative_structure,
                    'propp_characters': json.dumps([
                        {
                            'name': char.character_name,
                            'type': char.character_type.value,
                            'reasoning': char.reasoning,
                            'actions': char.key_actions
                        }
                        for char in analysis.characters
                    ]),
                    'num_characters': len(analysis.characters)
                }
                
            elif analysis_type == "booker":
                analysis = classificador.invoke(prompt.format(texto=row['text']))
                result = {
                    'story_id': row['id'],
                    'title': row['title'],
                    'booker_plot': analysis.plot_type.value,
                    'booker_reasoning': analysis.reasoning,
                    'booker_confidence': analysis.confidence
                }
                
            else:  # complete
                analysis = analyze_complete(row['text'])
                result = {
                    'story_id': row['id'],
                    'title': row['title'],
                    # Booker
                    'booker_plot': analysis.booker_plot.value,
                    'booker_reasoning': analysis.booker_reasoning,
                    'booker_confidence': analysis.booker_confidence,
                    # Propp
                    'propp_confidence': analysis.propp_confidence,
                    'propp_narrative': analysis.propp_narrative,
                    'propp_characters': json.dumps([
                        {
                            'name': char.character_name,
                            'type': char.character_type.value,
                            'reasoning': char.reasoning,
                            'actions': char.key_actions
                        }
                        for char in analysis.propp_characters
                    ]),
                    'num_characters': len(analysis.propp_characters),
                    # Integrada
                    'integrated_analysis': analysis.integrated_analysis
                }
            
            results.append(result)
            print(f"✅ Concluído com sucesso")
            
        except Exception as e:
            print(f"❌ Erro ao processar {row['title']}: {e}")
            continue
    
    # Criar DataFrame
    results_df = pd.DataFrame(results)
    
    # Salvar no banco
    table_name = f'story_analysis_{analysis_type}'
    results_df.to_sql(table_name, conn, if_exists='replace', index=False)
    
    conn.close()
    
    print(f"\n{'='*80}")

### Question:
Pode reconsiderar isso usando langchain?

### Answer:
Claro! Vou refatorar usando LangChain de forma mais idiomática, aproveitando melhor suas funcionalidades como chains, prompts templates e output parsers:

```python
# File: /Users/gustavosarti/Work/code/xer_persona/xer_persona/notebooks/classificar_contos.ipynb

import pandas as pd
import sqlite3
from pathlib import Path
from pydantic import BaseModel, Field
from enum import Enum
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough

# Definição dos Sete Enredos Básicos de Christopher Booker
SEVEN_BASIC_PLOTS = {
    "overcoming_the_monster": {
        "name": "Overcoming the Monster",
        "description": "O herói enfrenta e derrota uma força ameaçadora (monstro, vilão, etc.)"
    },
    "rags_to_riches": {
        "name": "Rags to Riches",
        "description": "O protagonista começa em circunstâncias humildes e alcança sucesso/riqueza"
    },
    "the_quest": {
        "name": "The Quest",
        "description": "O herói parte em uma jornada para alcançar um objetivo importante"
    },
    "voyage_and_return": {
        "name": "Voyage and Return",
        "description": "O protagonista viaja para um lugar estranho e retorna transformado"
    },
    "comedy": {
        "name": "Comedy",
        "description": "Confusão e mal-entendidos são resolvidos, levando a um final feliz"
    },
    "tragedy": {
        "name": "Tragedy",
        "description": "O protagonista tem uma falha fatal que leva à sua queda"
    },
    "rebirth": {
        "name": "Rebirth",
        "description": "O protagonista é transformado ou redimido após eventos negativos"
    }
}

# Definição dos 7 personagens de Propp
PROPP_CHARACTERS = {
    "villain": {
        "name": "Villain (Antagonista)",
        "description": "O vilão que luta contra o herói, causa dano ou prejudica alguém"
    },
    "donor": {
        "name": "Donor (Doador)",
        "description": "Prepara o herói ou lhe dá algum objeto mágico/ajuda"
    },
    "helper": {
        "name": "Helper (Auxiliar)",
        "description": "Ajuda o herói em sua busca, geralmente após receber o objeto mágico"
    },
    "princess": {
        "name": "Princess (Princesa) and her Father",
        "description": "O objeto da busca do herói, frequentemente oferecida como recompensa; seu pai frequentemente propõe tarefas difíceis"
    },
    "dispatcher": {
        "name": "Dispatcher (Mandante)",
        "description": "Envia o herói em sua missão através de uma ordem ou pedido"
    },
    "hero": {
        "name": "Hero (Herói)",
        "description": "O protagonista que reage ao doador, parte em busca e se casa com a princesa"
    },
    "false_hero": {
        "name": "False Hero (Falso Herói)",
        "description": "Toma crédito pelas ações do herói ou tenta casar com a princesa"
    }
}

# ============================================================================
# SCHEMAS PYDANTIC
# ============================================================================

class PlotType(str, Enum):
    overcoming_the_monster = "overcoming_the_monster"
    rags_to_riches = "rags_to_riches"
    the_quest = "the_quest"
    voyage_and_return = "voyage_and_return"
    comedy = "comedy"
    tragedy = "tragedy"
    rebirth = "rebirth"

class ProppCharacterType(str, Enum):
    villain = "villain"
    donor = "donor"
    helper = "helper"
    princess = "princess"
    dispatcher = "dispatcher"
    hero = "hero"
    false_hero = "false_hero"

class PlotClassification(BaseModel):
    """Classificação de enredo segundo Booker"""
    plot_type: PlotType = Field(description="O tipo de enredo identificado")
    reasoning: str = Field(description="Explicação detalhada do motivo da classificação")
    confidence: float = Field(description="Confiança da classificação (0-1)")
    key_elements: List[str] = Field(description="Elementos-chave da história que justificam a classificação")

class ProppCharacter(BaseModel):
    """Personagem individual segundo Propp"""
    character_name: str = Field(description="Nome ou descrição do personagem na história")
    character_type: ProppCharacterType = Field(description="Tipo de personagem segundo Propp")
    reasoning: str = Field(description="Justificativa para a classificação deste personagem")
    key_actions: List[str] = Field(description="Ações principais que justificam a classificação")

class ProppAnalysis(BaseModel):
    """Análise completa de personagens segundo Propp"""
    characters: List[ProppCharacter] = Field(description="Lista de personagens identificados")
    narrative_structure: str = Field(description="Descrição da estrutura narrativa segundo Propp")
    confidence: float = Field(description="Confiança geral da análise (0-1)")
    narrative_functions: List[str] = Field(description="Funções narrativas de Propp presentes na história")

class CompleteStoryAnalysis(BaseModel):
    """Análise literária completa: Booker + Propp"""
    # Análise de Booker
    booker_plot: PlotType = Field(description="Tipo de enredo segundo Booker")
    booker_reasoning: str = Field(description="Justificativa da classificação de Booker")
    booker_confidence: float = Field(description="Confiança da classificação de Booker (0-1)")
    booker_key_elements: List[str] = Field(description="Elementos-chave segundo Booker")
    
    # Análise de Propp
    propp_characters: List[ProppCharacter] = Field(description="Personagens identificados segundo Propp")
    propp_narrative: str = Field(description="Estrutura narrativa segundo Propp")
    propp_confidence: float = Field(description="Confiança da análise de Propp (0-1)")
    propp_functions: List[str] = Field(description="Funções narrativas de Propp")
    
    # Análise integrada
    integrated_analysis: str = Field(description="Como as análises de Booker e Propp se complementam")
    story_archetype: str = Field(description="Arquétipo geral da história baseado em ambas análises")

# ============================================================================
# CONFIGURAÇÃO DO MODELO
# ============================================================================

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# ============================================================================
# PROMPTS TEMPLATES
# ============================================================================

# Template para análise de Booker
booker_template = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em análise literária, especificamente nos Sete Enredos Básicos de Christopher Booker.
    
Analise a história fornecida e classifique-a em um dos sete enredos básicos:

{plot_descriptions}

Seja preciso e detalhado em sua análise. Identifique os elementos-chave que justificam sua classificação.

{format_instructions}"""),
    ("human", "História para análise:\n\n{texto}")
])

# Template para análise de Propp
propp_template = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em morfologia do conto folclórico de Vladimir Propp.

Analise os personagens da história segundo a morfologia de Propp:

{character_descriptions}

Para cada personagem identificado:
1. Determine qual tipo de personagem de Propp ele representa
2. Identifique as ações específicas que justificam essa classificação
3. Explique como ele contribui para a estrutura narrativa

Importante:
- Um personagem pode desempenhar múltiplos papéis
- Nem todos os 7 tipos precisam estar presentes
- Foque nas funções narrativas, não apenas em características superficiais
- Identifique as funções narrativas de Propp presentes (ex: ausência, interdição, transgressão, etc.)

{format_instructions}"""),
    ("human", "História para análise:\n\n{texto}")
])

# Template para análise completa
complete_template = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em análise literária, combinando as perspectivas de Christopher Booker e Vladimir Propp.

Realize uma análise literária completa da história:

## 1. ANÁLISE DE BOOKER - Os Sete Enredos Básicos:
{plot_descriptions}

## 2. ANÁLISE DE PROPP - Morfologia dos Personagens:
{character_descriptions}

## 3. ANÁLISE INTEGRADA
Explique como as duas perspectivas se complementam para entender a história.
Identifique o arquétipo geral da história baseado em ambas análises.

Seja preciso, detalhado e acadêmico em suas análises.

{format_instructions}"""),
    ("human", "História para análise:\n\n{texto}")
])

# ============================================================================
# CHAINS LANGCHAIN
# ============================================================================

def create_booker_chain():
    """Cria chain para análise de Booker"""
    parser = PydanticOutputParser(pydantic_object=PlotClassification)
    
    chain = (
        {
            "texto": RunnablePassthrough(),
            "plot_descriptions": lambda _: "\n".join(
                f"- {k}: {v['description']}" 
                for k, v in SEVEN_BASIC_PLOTS.items()
            ),
            "format_instructions": lambda _: parser.get_format_instructions()
        }
        | booker_template
        | llm
        | parser
    )
    
    return chain

def create_propp_chain():
    """Cria chain para análise de Propp"""
    parser = PydanticOutputParser(pydantic_object=ProppAnalysis)
    
    chain = (
        {
            "texto": RunnablePassthrough(),
            "character_descriptions": lambda _: "\n".join(
                f"- {k}: {v['description']}" 
                for k, v in PROPP_CHARACTERS.items()
            ),
            "format_instructions": lambda _: parser.get_format_instructions()
        }
        | propp_template
        | llm
        | parser
    )
    
    return chain

def create_complete_chain():
    """Cria chain para análise completa"""
    parser = PydanticOutputParser(pydantic_object=CompleteStoryAnalysis)
    
    chain = (
        {
            "texto": RunnablePassthrough(),
            "plot_descriptions": lambda _: "\n".join(
                f"- {k}: {v['description']}" 
                for k, v in SEVEN_BASIC_PLOTS.items()
            ),
            "character_descriptions": lambda _: "\n".join(
                f"- {k}: {v['description']}" 
                for k, v in PROPP_CHARACTERS.items()
            ),
            "format_instructions": lambda _: parser.get_format_instructions()
        }
        | complete_template
        | llm
        | parser
    )
    
    return chain

# ============================================================================
# FUNÇÕES DE ANÁLISE
# ============================================================================

# Criar chains globais
booker_chain = create_booker_chain()
propp_chain =

### Question:
retome... reescreva a partir das chains langchain